Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

## Prompt templates and parsers

- A `PromptTemplate` is a prompt with named holes to fill in
- A parser turns the reply into the type we want, here a plain string
- `prompt | llm | parser` pipes each stage into the next

Two small pieces that every chain later in the course is built from.

### Exercise PromptTemplate 
Define a template and render the message. 
Complete the PromptTemplate definition and render the finished prompt.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()

# 1) Define the template input variables (they should include 'topic' and 'tone')
input_variables = ["topic", "tone"]

# 2) Define the template content so that it uses {topic} and {tone}
template = """Write a short paragraph about the topic: {topic}.
Keep the tone of the response: {tone}."""

prompt = PromptTemplate(input_variables=____, template=____) # <- fill in the variables here

# 3) Render the finished prompt by substituting the values
rendered = prompt.format(topic="machine learning", tone="enthusiastic")

print(rendered)
assert "{topic}" not in rendered and "{tone}" not in rendered, "It looks like you didn't format the template."
```

In [1]:
from langchain_core.prompts import PromptTemplate

# 1) Define the template input variables (they should include 'topic' and 'tone')
input_variables = ["topic", "tone"]

# 2) Define the template content so that it uses {topic} and {tone}
template = """Write a short paragraph about the topic: {topic}.
Keep the tone of the response: {tone}."""

# 3) Fill in the two blanks: pass the list and the string defined above
prompt = PromptTemplate(input_variables=input_variables, template=template)

# 4) Render the finished prompt by substituting the values
rendered = prompt.format(topic="machine learning", tone="enthusiastic")

print(rendered)
assert "{topic}" not in rendered and "{tone}" not in rendered, \
    "It looks like you didn't format the template."

Write a short paragraph about the topic: machine learning.
Keep the tone of the response: enthusiastic.


### Exercise OutputParser 
Add a simple parser that forces the return of a plain string. Fill in "____" to build a correct LCEL chain.

#### Optional : write it first

The next cell is the finished version ; nothing below depends on doing this first.

```python
from langchain_core.output_parsers import ____.

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

template = "Give a definition of the term: {term} in one sentence."
prompt = PromptTemplate.from_template(template)

llm = make_llm() 

parser = StrOutputParser()

chain = ____ | ____ | ____  # Assemble the chain prompt -> llm -> parser

chain.invoke({"term": "AGI"})
```

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

template = "Give a definition of the term: {term} in one sentence."
prompt = PromptTemplate.from_template(template)

llm = make_llm()
parser = StrOutputParser()

# Assemble the chain: prompt -> llm -> parser
chain = prompt | llm | parser

print(chain.invoke({"term": "AGI"}))

Artificial General Intelligence (AGI) refers to a hypothetical artificial intelligence system that possesses the ability to understand, learn, and apply knowledge across a wide range of tasks, similar to human intelligence, with the potential to surpass human capabilities in various domains.


### Try a missing variable

- Call `prompt.format()` and leave one of the variables out
- The error names the variable. Templates fail early, which is the point of
  them ; a hand-built f-string would have sent the model a broken prompt